In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")

In [16]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  # [!code highlight]
from langchain_core.tools import tool

@tool("get_user_info", description = "获取用户信息",return_direct=True)
def get_user_info(messages: list[dict]) -> str:
    """
    获取用户信息的工具函数
    """
    for message in messages:
        if message["role"] == "user":
            return f"用户说：{message['content']}"
    return "没有找到用户信息"

agent = create_agent(
    "deepseek-chat",
    [get_user_info],
    checkpointer=InMemorySaver(),  # [!code highlight]
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "你好我是一名伟大的大模型应用工程师，虽然现在不是，但是我会努力成为的"}]},
    {"configurable": {"thread_id": "1"}},  # [!code highlight]
)

print(result)

{'messages': [HumanMessage(content='你好我是一名伟大的大模型应用工程师，虽然现在不是，但是我会努力成为的', additional_kwargs={}, response_metadata={}, id='a9d909a8-c83c-4744-a6aa-57d3c57f87d4'), AIMessage(content='你好！👋 看到你这样的目标和热情，真的很让人振奋！\n\n**现在的你不是，不代表未来的你不是。** 能够立下"成为伟大的大模型应用工程师"这个目标，本身就已经是迈出了最重要的一步。\n\n我特别欣赏你这句话——**"虽然现在不是，但是我会努力成为的"**。这种坦诚面对现状、同时坚定向前的态度，正是真正优秀工程师的底色。\n\n在你朝着这个目标前进的路上，我作为 AI 助手，可以帮你做这些事情：\n\n🚀 **学习路线**\n- 帮你规划从入门到精通的路径（Python、Prompt Engineering、RAG、Agent、Fine-tuning...）\n- 推荐高质量的课程、论文、开源项目\n\n💡 **实战项目**\n- 一起设计大模型应用架构\n- 帮你 debug、优化代码\n- 探讨如何将 LLM 落地到真实场景\n\n📚 **深入原理**\n- 解读 Transformer、Attention 机制\n- 分析各种模型的优缺点\n- 讨论 AI 工程化的最佳实践\n\n---\n\n**要不要告诉我，你现在的基础怎么样？** 比如编程水平、对 AI 的了解程度，或者有没有特别想做的项目方向？我们可以一起制定一个属于你的成长计划！💪\n\n未来伟大的大模型应用工程师，我很期待陪你一起见证你的成长！✨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 271, 'prompt_tokens': 303, 'total_tokens': 574, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'pro

In [8]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver


class CustomAgentState(AgentState):  # [!code highlight]
    user_id: str  # [!code highlight]
    preferences: dict  # [!code highlight]

agent = create_agent(
    "deepseek-chat",
    [get_user_info],
    state_schema=CustomAgentState,  # [!code highlight]
    checkpointer=InMemorySaver(),
)

# Custom state can be passed in invoke
result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "Hello"}],
        "user_id": "user_123",  # [!code highlight]
        "preferences": {"theme": "dark"}  # [!code highlight]
    },
    {"configurable": {"thread_id": "1"}})

print(result)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='be32945b-b607-4def-9433-4c331ca23a8e'), AIMessage(content='你好！我是 DeepSeek，很高兴为你服务！😊\n\n请问有什么可以帮你的吗？我可以回答你的问题、提供帮助，或者陪你聊天。有什么想聊的，尽管说！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 286, 'total_tokens': 327, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 30}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '5831ed5c-de55-43ff-8709-d64810ae8d1f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e25db-f7bd-7693-9c4f-869c98faf198-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 286, 'output_tokens': 41, 'total_tokens': 327, 'input_token_details': {'cache_read': 256}, 'output_token_details': 

In [ ]:
%pip install gradio

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  # [!code highlight]
from langchain_core.tools import tool
from langchain.tools import ToolRuntime

class CustomAgentState(AgentState):  # [!code highlight]
    user_id: str  # [!code highlight]

@tool("get_user_info", description = "获取用户信息",return_direct=True)
def get_user_info(
    runtime:ToolRuntime
    ) -> str:
    """
    获取用户信息的工具函数
    """
    user_id = runtime.state["user_id"]
    return "User is John Smith" if user_id == "user_123" else "Unknown user"

agent = create_agent(
    "deepseek-chat",
    [get_user_info],
    checkpointer=InMemorySaver(),  # [!code highlight]
    state_schema=CustomAgentState
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "你好我是一名伟大的大模型应用工程师，虽然现在不是，但是我会努力成为的"}]},
    {"configurable": {"thread_id": "1"}},  # [!code highlight]
)

print(result)

TypeError: Pregel.invoke() takes from 2 to 3 positional arguments but 4 were given

In [23]:
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime


class CustomState(AgentState):
    user_id: str

@tool
def get_user_info(
    runtime: ToolRuntime
) -> str:
    """Look up user info."""
    user_id = runtime.state["user_id"]
    return "User is John Smith" if user_id == "user_123" else "Unknown user"

agent = create_agent(
    model="deepseek-chat",
    tools=[get_user_info],
    state_schema=CustomState,
)

result = agent.invoke({
    "messages": "look up user information",
    "user_id": "user_123"
})
print(result["messages"][-1].content)
# > User is John Smith.

The user's name is **John Smith**. Let me know if you'd like any further information or assistance!
